# Damped oscillator using the finite-difference method



#### <p style="text-align: right;"> &#9989; **put your name here** </p>

---
-- By GPT 5.2

We study the **damped oscillator**

$$
m\ddot{x} + c\dot{x} + kx = 0,
$$

where

- $m$ is the mass,
- $c$ is the damping coefficient,
- $k$ is the stiffness,
- $x(t)$ is the displacement.

This model appears in many materials contexts:

- atomic vibrations,
- defect motion,
- damped lattice or micro-mechanical oscillations,
- viscoelastic response.

We will:

1. Rewrite the governing equation
2. Derive a finite-difference update formula
3. Implement the scheme in Python
4. Study underdamped, critically damped, and overdamped cases
5. Compare with the analytical solution for validation

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Governing equation

The damped oscillator is

$$
m\ddot{x} + c\dot{x} + kx = 0.
$$

Divide by $m$:

$$
\ddot{x} + \frac{c}{m}\dot{x} + \frac{k}{m}x = 0.
$$

It is often useful to define

$$
\omega_n = \sqrt{\frac{k}{m}},
\qquad
\zeta = \frac{c}{2\sqrt{mk}},
$$

where

- $\omega_n$ is the natural frequency,
- $\zeta$ is the damping ratio.

Then the behavior is:

- **underdamped** if $\zeta < 1$
- **critically damped** if $\zeta = 1$
- **overdamped** if $\zeta > 1$

In [ ]:
def damping_ratio(m, c, k):
    return c / (2 * np.sqrt(m * k))

def natural_frequency(m, k):
    return np.sqrt(k / m)

## 3. Finite-difference approximation

We approximate the derivatives at time step $n$ by central differences:

$$
\ddot{x}(t_n) \approx \frac{x^{n+1} - 2x^n + x^{n-1}}{\Delta t^2},
$$

$$
\dot{x}(t_n) \approx \frac{x^{n+1} - x^{n-1}}{2\Delta t}.
$$

Substitute into

$$
m\ddot{x} + c\dot{x} + kx = 0:
$$

$$
m\frac{x^{n+1} - 2x^n + x^{n-1}}{\Delta t^2}
+
c\frac{x^{n+1} - x^{n-1}}{2\Delta t}
+
k x^n = 0.
$$

Collect the $x^{n+1}$ terms:

$$
\left(\frac{m}{\Delta t^2} + \frac{c}{2\Delta t}\right)x^{n+1}
+
\left(-\frac{2m}{\Delta t^2} + k\right)x^n
+
\left(\frac{m}{\Delta t^2} - \frac{c}{2\Delta t}\right)x^{n-1}
= 0.
$$

So the update formula is

$$
x^{n+1}
=
\frac{
\left(\frac{2m}{\Delta t^2}-k\right)x^n
-
\left(\frac{m}{\Delta t^2}-\frac{c}{2\Delta t}\right)x^{n-1}
}{
\frac{m}{\Delta t^2}+\frac{c}{2\Delta t}
}.
$$

## 4. Initial conditions

To march forward in time, we need:

- $x(0) = x_0$
- $\dot{x}(0) = v_0$

The finite-difference scheme needs $x^0$ and $x^1$.

We take

$$
x^0 = x_0.
$$

A second-order Taylor expansion gives

$$
x^1 \approx x_0 + v_0\Delta t + \frac{1}{2}a_0 \Delta t^2,
$$

where

$$
a_0 = \ddot{x}(0) = -\frac{c}{m}v_0 - \frac{k}{m}x_0.
$$

In [ ]:
def initial_acceleration(m, c, k, x0, v0):
    return -(c/m) * v0 - (k/m) * x0

## 5. Finite-difference solver

In [ ]:
def damped_oscillator_fd(m, c, k, x0, v0, dt, t_end):
    Nt = int(np.round(t_end / dt)) + 1
    t = np.linspace(0, dt*(Nt-1), Nt)
    x = np.zeros(Nt)

    # initial conditions
    x[0] = x0
    a0 = initial_acceleration(m, c, k, x0, v0)
    x[1] = ??

    A = m/dt**2 + c/(2*dt)
    B = ??
    C = ??

    for n in range(1, Nt-1):
        x[n+1] = (B*x[n] - C*x[n-1]) / A

    return t, x

## 6. Test case

In [ ]:
m = 1.0
k = 100.0
c = 2.0
x0 = 1.0
v0 = 0.0
dt = 0.001
t_end = 2.0

t, x = damped_oscillator_fd(m, c, k, x0, v0, dt, t_end)

print("Natural frequency =", natural_frequency(m, k))
print("Damping ratio =", damping_ratio(m, c, k))

In [ ]:
plt.plot(t, x, '.')
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Finite-difference solution of damped oscillator")
plt.grid(True)
plt.show()

## 7. Analytical solution for comparison

The exact solution depends on the damping ratio \(\zeta\).

### Underdamped: $\zeta < 1$

$$
x(t) = e^{-\zeta\omega_n t}
\left[
A\cos(\omega_d t) + B\sin(\omega_d t)
\right],
$$

where

$$
\omega_d = \omega_n\sqrt{1-\zeta^2}.
$$

### Critically damped: $\zeta = 1$

$$
x(t) = (A + Bt)e^{-\omega_n t}.
$$

### Overdamped: $\zeta > 1$

$$
x(t) = A e^{r_1 t} + B e^{r_2 t},
$$

where $r_1$ and $r_2$ are the two real roots.

In [ ]:
def damped_oscillator_exact(m, c, k, x0, v0, t):
    wn = np.sqrt(k / m)
    zeta = c / (2 * np.sqrt(m * k))

    if zeta < 1 - 1e-12:
        wd = ??
        A = ??
        B = (v0 + zeta * wn * x0) / wd
        return np.exp(-zeta * wn * t) * (A * np.cos(wd * t) + B * np.sin(wd * t))

    elif abs(zeta - 1) <= 1e-12:
        A = x0
        B = v0 + wn * x0
        return (A + B * t) * np.exp(-wn * t)

    else:
        s = np.sqrt(zeta**2 - 1)
        r1 = -wn * (zeta - s)
        r2 = -wn * (zeta + s)
        A = (v0 - r2 * x0) / (r1 - r2)
        B = x0 - A
        return A * np.exp(r1 * t) + B * np.exp(r2 * t)

In [ ]:
x_exact = damped_oscillator_exact(m, c, k, x0, v0, t)

plt.plot(t, x_exact, label="exact", lw=2)
plt.plot(t, x, '--', label="finite difference")
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Exact vs finite-difference solution")
plt.grid(True)
plt.legend()
plt.show()

print("Maximum absolute error =", np.max(np.abs(x - x_exact)))

## 8. Underdamped, critically damped, and overdamped cases

We now compare the three regimes.

For fixed $m$ and $k$, the critical damping is

$$
c_{crit} = 2\sqrt{mk}.
$$

In [ ]:
m = 1.0
k = 25.0
x0 = 1.0
v0 = 0.0
dt = 0.002
t_end = 4.0

c_crit = 2 * np.sqrt(m * k)
print("Critical damping coefficient =", c_crit)

cases = [
    ("Underdamped", 0.4 * c_crit),
    ("Critical", 1.0 * c_crit),
    ("Overdamped", 2.0 * c_crit),
]

for ??, ?? in cases:
    t, x_fd = damped_oscillator_fd(m, c, k, x0, v0, dt, t_end)
    plt.plot(t, x_fd, label=f"{label}, zeta={damping_ratio(m,c,k):.2f}")

plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Three damping regimes")
plt.grid(True)
plt.legend()
plt.show()

Interpretation:

- **Underdamped:** oscillates while decaying
- **Critical damping:** returns to equilibrium as fast as possible without oscillating
- **Overdamped:** no oscillation, but slower return than critical damping

## 9. Effect of time step

A finite-difference method is accurate only when the time step is sufficiently small.

Let us compare several values of $\Delta t$.

In [ ]:
m = 1.0
k = 100.0
c = 2.0
x0 = 1.0
v0 = 0.0
t_end = 2.0

dt_list = [0.1, 0.05, 0.01]

t_fine = np.linspace(0, t_end, 2000)
x_ref = damped_oscillator_exact(m, c, k, x0, v0, t_fine)

plt.plot(t_fine, x_ref, 'k-', lw=2, label="exact")

for ?? in dt_list:
    t_num, x_num = damped_oscillator_fd(m, c, k, x0, v0, dt, t_end)
    plt.plot(t_num, x_num, 'o--', ms=3, label=f"FD dt={dt}")

plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Effect of time step on finite-difference accuracy")
plt.grid(True)
plt.legend()
plt.show()

## 10. Phase portrait

A useful way to understand oscillator dynamics is to plot velocity versus displacement.

We estimate velocity from a central difference:

$$
v^n \approx \frac{x^{n+1}-x^{n-1}}{2\Delta t}.
$$

In [ ]:
m = 1.0
k = 25.0
c = 2.0
x0 = 1.0
v0 = 0.0
dt = 0.002
t_end = 6.0

t, x = damped_oscillator_fd(m, c, k, x0, v0, dt, t_end)

v = np.zeros_like(x)
v[0] = v0

v[1:-1] = ??
v[-1] = (x[-1] - x[-2]) / dt

plt.plot(x, v)
plt.xlabel("x")
plt.ylabel("v")
plt.title("Phase portrait of damped oscillator")
plt.grid(True)
plt.show()

## 11. Energy decay

The total mechanical energy is

$$
E(t) = \frac{1}{2}m\dot{x}^2 + \frac{1}{2}kx^2.
$$

Damping removes energy from the system, so the energy should decrease with time.

In [ ]:
m = 1.0
k = 25.0
c = 2.0
x0 = 1.0
v0 = 0.0
dt = 0.002
t_end = 6.0

t, x = damped_oscillator_fd(m, c, k, x0, v0, dt, t_end)

v = np.zeros_like(x)
v[0] = v0
v[1:-1] = ??
v[-1] = (x[-1] - x[-2]) / dt

E = ??

plt.plot(t, E)
plt.xlabel("t")
plt.ylabel("E(t)")
plt.title("Energy decay in the damped oscillator")
plt.grid(True)
plt.show()

## 12. Summary

In this notebook, we:

- derived a central-difference finite-difference scheme for
  $$
  m\ddot{x}+c\dot{x}+kx=0,
  $$
- implemented the update formula,
- used Taylor expansion to initialize the second-order method,
- compared the finite-difference result to the exact analytical solution,
- explored underdamped, critical, and overdamped behavior,
- examined phase portraits and energy decay.

This model is a fundamental example for vibrations, defects, and relaxation phenomena in materials science.

## 13. Optional exercises

1. Change the initial velocity and study how the motion changes.
2. Find a time step that makes the numerical solution inaccurate.
3. Compare weak damping and strong damping.
4. Derive a backward-difference or Newmark-type scheme.
5. Add an external force $F(t)$ and study the driven oscillator.